# 04c · Modèle 3 : segmentation des clients, **tous les algorithmes de clustering**
**Projet : E-Commerce Intelligence Platform** · CRISP-DM, phase 4 (Modeling) et phase 5 (Evaluation)

**Objectif** : regrouper les clients en segments homogènes selon leur **récence**, leur **valeur**, leur **satisfaction** et leur **expérience de livraison**, pour orienter la fidélisation et le marketing.

**Démarche**
1. Variables de clustering (déjà transformées en phase 3), plafonnement des valeurs extrêmes.
2. Choix du nombre de clusters *k* avec K-Means : coude, silhouette, Davies-Bouldin, Calinski-Harabasz, **stabilité** (ARI) ; avis du **BIC** (mélange gaussien).
3. **Tous les algorithmes de clustering** sont comparés au même *k* (ou sans *k* pour les méthodes à densité) : K-Means, MiniBatch K-Means, Bisecting K-Means, mélanges gaussiens (full, diag, bayésien), Birch, hiérarchique (Ward, average, complete), spectral, Affinity Propagation, Mean Shift, DBSCAN, HDBSCAN, OPTICS.
4. Choix de l'algorithme final parmi ceux qui peuvent **affecter de nouveaux clients**, vérification de sa stabilité.
5. Profils, **noms métier** et **actions recommandées**, vue « retard vécu et satisfaction ».

> Contexte : 97 % des clients n'achètent qu'une fois. La fréquence n'est donc pas une variable de segmentation : les segments décrivent des **profils d'achat et d'expérience**, pas une fidélité dans la durée.
> Données : dataset Kaggle `e-commerce-dataset-model-3`. Durée indicative : 10 à 25 minutes.

## 1. Imports et paramètres

In [ ]:
import os, glob, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.base import clone
from sklearn.cluster import (KMeans, MiniBatchKMeans, AgglomerativeClustering, Birch, DBSCAN, OPTICS,
                             MeanShift, estimate_bandwidth, AffinityPropagation, SpectralClustering)
try:
    from sklearn.cluster import BisectingKMeans; HAS_BISECT = True
except Exception:
    HAS_BISECT = False
try:
    from sklearn.cluster import HDBSCAN; HAS_HDBSCAN = True
except Exception:
    HAS_HDBSCAN = False
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score, adjusted_rand_score
from sklearn.mixture import GaussianMixture, BayesianGaussianMixture
from sklearn.neighbors import NearestNeighbors

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60); pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

SEED = 42
rng = np.random.default_rng(SEED)
FAST = os.environ.get("FAST", "0") == "1"
OUT_DIR = os.environ.get("OUT_DIR", "/kaggle/working/models")
os.makedirs(OUT_DIR, exist_ok=True)

K_RANGE = range(2, 8) if FAST else range(2, 11)
K_FINAL = None              # None : choix automatique ; sinon un entier
FINAL_ALGO = None           # None : choix automatique parmi les algorithmes assignables ; sinon un nom du tableau
MIN_CLUSTER_SHARE = 1.0     # un segment doit représenter au moins 1 % des clients de l'échantillon (évite les micro-clusters)
CLIP_Q = 0.005              # plafonnement aux quantiles 0,5 % et 99,5 %
SIL_SAMPLE = 3000 if FAST else 15000
N_INIT = 3 if FAST else 10
N_BOOT = 2 if FAST else 5
EVAL_N = 1200 if FAST else 5000          # taille de l'échantillon de comparaison des algorithmes
AP_N = 600 if FAST else 2000             # Affinity Propagation : échantillon plus petit (coût en O(n²))

# Chemins des jeux de données Kaggle (créés à partir de la sortie du notebook 03)
DATA_PATHS = {
    "model1_delivery_duration.csv": "/kaggle/input/datasets/oussamafriha/e-commerce-dataset-model-1/model1_delivery_duration.csv",
    "model2_late_risk.csv": "/kaggle/input/datasets/oussamafriha/e-commerce-dataset-model-2/model2_late_risk.csv",
    "model3_customer_segmentation.csv": "/kaggle/input/datasets/oussamafriha/e-commerce-dataset-model-3/model3_customer_segmentation.csv",
}

def find_file(name):
    """DATA_DIR (si défini), puis le chemin Kaggle ci-dessus, puis une recherche dans /kaggle/input."""
    cands = []
    if os.environ.get("DATA_DIR"):
        cands.append(os.path.join(os.environ["DATA_DIR"], name))
    cands.append(DATA_PATHS.get(name, ""))
    cands += glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    for c in cands:
        if c and os.path.exists(c):
            return c
    raise FileNotFoundError(f"{name} introuvable : ajoute le dataset correspondant en entrée du notebook (Add Input).")

## 2. Chargement et variables de clustering

In [ ]:
m3 = pd.read_csv(find_file("model3_customer_segmentation.csv"), parse_dates=["first_purchase", "last_purchase"])
CLUSTER_FEATURES = ["recency_days", "log_monetary", "avg_review_score", "avg_delivery_days", "log_max_days_late", "avg_gap_days"]
RS = [c + "_rs" for c in CLUSTER_FEATURES]
assert not [c for c in CLUSTER_FEATURES + RS + ["customer_unique_id", "region", "customer_state"] if c not in m3.columns]
print(f"{len(m3):,} clients | à achat répété : {m3['is_repeat_customer'].mean()*100:.2f} % | sans avis : {m3['review_missing'].mean()*100:.2f} %")
display(m3[CLUSTER_FEATURES].describe().round(2))

## 3. Plafonnement des valeurs extrêmes
Les variables sont déjà mises à l'échelle de façon robuste, mais il reste des extrêmes (écart promesse / réel de -189 à +146 jours). Les algorithmes à distances y sont sensibles : on plafonne chaque variable aux quantiles `CLIP_Q` et `1 - CLIP_Q`.

In [ ]:
X_raw = m3[RS].copy()
lo, hi = X_raw.quantile(CLIP_Q), X_raw.quantile(1 - CLIP_Q)
X = X_raw.clip(lower=lo, upper=hi, axis=1)
Xv = X.values
display(pd.DataFrame({"min avant": X_raw.min(), "min après": X.min(), "max avant": X_raw.max(), "max après": X.max(),
                      "clients plafonnés (%)": (X_raw != X).mean() * 100}))
fig, ax = plt.subplots(1, 2, figsize=(16, 4))
sns.heatmap(m3[CLUSTER_FEATURES].corr(method="spearman"), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax[0])
ax[0].set_title("Corrélations de Spearman entre variables de clustering")
X.boxplot(ax=ax[1], rot=30); ax[1].set_title("Variables mises à l'échelle et plafonnées")
plt.tight_layout(); plt.show()

## 4. Choix du nombre de clusters (K-Means)
Pour chaque *k* : inertie (coude), **silhouette** (échantillon), **Davies-Bouldin** (plus bas = mieux), **Calinski-Harabasz** (plus haut = mieux) et **stabilité** (ARI moyen entre la partition de référence et celles de sous-échantillons de 80 % ; proche de 1 = segments reproductibles).

In [ ]:
sil_idx = rng.choice(len(Xv), size=min(SIL_SAMPLE, len(Xv)), replace=False)
rows = []
for k in K_RANGE:
    km = KMeans(n_clusters=k, n_init=N_INIT, random_state=SEED).fit(Xv)
    lab = km.labels_
    aris = []
    for b in range(N_BOOT):
        sub = rng.choice(len(Xv), size=int(0.8 * len(Xv)), replace=False)
        aris.append(adjusted_rand_score(lab, KMeans(n_clusters=k, n_init=3, random_state=SEED + b + 1).fit(Xv[sub]).predict(Xv)))
    rows.append({"k": k, "inertie": km.inertia_, "silhouette": silhouette_score(Xv[sil_idx], lab[sil_idx]),
                 "davies_bouldin": davies_bouldin_score(Xv, lab), "calinski_harabasz": calinski_harabasz_score(Xv, lab),
                 "stabilité_ARI": float(np.mean(aris)), "taille_min_%": np.bincount(lab).min() / len(lab) * 100})
sel = pd.DataFrame(rows).set_index("k")
display(sel)
fig, ax = plt.subplots(1, 4, figsize=(20, 4))
sel["inertie"].plot(marker="o", ax=ax[0], title="Coude : inertie")
sel["silhouette"].plot(marker="o", ax=ax[1], title="Silhouette (plus haut = mieux)", color="tab:green")
sel["davies_bouldin"].plot(marker="o", ax=ax[2], title="Davies-Bouldin (plus bas = mieux)", color="tab:red")
sel["stabilité_ARI"].plot(marker="o", ax=ax[3], title="Stabilité (ARI moyen)", color="tab:purple")
plt.tight_layout(); plt.show()

## 5. Avis du critère BIC (mélange gaussien)

In [ ]:
g_idx = rng.choice(len(Xv), size=min(3000 if FAST else 30000, len(Xv)), replace=False)
bic = pd.Series({k: GaussianMixture(n_components=k, covariance_type="full", random_state=SEED).fit(Xv[g_idx]).bic(Xv[g_idx])
                 for k in K_RANGE}, name="BIC")
bic.plot(marker="o", figsize=(7, 3.5), title="BIC du mélange gaussien (échantillon)"); plt.show()
print("k minimisant le BIC :", int(bic.idxmin()))

## 6. Choix de *k*
Parmi les *k* de 3 à 8 dont la **stabilité** est d'au moins 0,70, on retient celui de **silhouette maximale** (sinon le plus stable). *k* = 2 est écarté, trop grossier pour des actions marketing. `K_FINAL` permet de forcer un choix après lecture des profils.

In [ ]:
cand = sel.loc[(sel.index >= 3) & (sel.index <= 8)]
stable = cand[cand["stabilité_ARI"] >= 0.70]
K = int(K_FINAL) if K_FINAL else int((stable if len(stable) else cand)["silhouette" if len(stable) else "stabilité_ARI"].idxmax())
print(f"Nombre de clusters retenu : k = {K}")
display(sel.loc[[K]])

## 7. Comparaison de **tous les algorithmes de clustering**
Tous les algorithmes sont ajustés sur le **même échantillon** de clients (`EVAL_N`), avec *k* = valeur retenue pour ceux qui en ont besoin. Les méthodes à densité (DBSCAN, HDBSCAN, OPTICS, Mean Shift, Affinity Propagation) trouvent seules leur nombre de groupes et peuvent isoler du **bruit** (étiquette -1).

Indicateurs : nombre de clusters, part de bruit, silhouette, Davies-Bouldin, Calinski-Harabasz, taille du plus petit groupe, **ARI contre K-Means** (accord entre méthodes). La colonne `assignable` indique si l'algorithme peut affecter de **nouveaux** clients.

In [ ]:
ev_idx = rng.choice(len(Xv), size=min(EVAL_N, len(Xv)), replace=False)
Xe = Xv[ev_idx]

# eps de DBSCAN : quantile 90 % de la distance au 12e voisin (heuristique du « coude » de la courbe des k-distances)
kd = NearestNeighbors(n_neighbors=12).fit(Xe).kneighbors(Xe)[0][:, -1]
EPS = float(np.quantile(kd, 0.90))
MIN_S = max(10, int(0.004 * len(Xe)))
BW = estimate_bandwidth(Xe, quantile=0.2, n_samples=min(1000, len(Xe)))

def algo_registry(k):
    A = {}
    def add(name, family, make, assign=False, n=None):
        A[name] = {"family": family, "make": make, "assign": assign, "n": n}
    add("K-Means", "Centroïdes", lambda: KMeans(n_clusters=k, n_init=N_INIT, random_state=SEED), True)
    add("MiniBatch K-Means", "Centroïdes", lambda: MiniBatchKMeans(n_clusters=k, n_init=N_INIT, batch_size=1024, random_state=SEED), True)
    if HAS_BISECT:
        add("Bisecting K-Means", "Centroïdes", lambda: BisectingKMeans(n_clusters=k, n_init=3, random_state=SEED), True)
    add("Mélange gaussien (full)", "Modèle probabiliste", lambda: GaussianMixture(n_components=k, covariance_type="full", n_init=2, random_state=SEED), True)
    add("Mélange gaussien (diag)", "Modèle probabiliste", lambda: GaussianMixture(n_components=k, covariance_type="diag", n_init=2, random_state=SEED), True)
    add("Mélange gaussien bayésien", "Modèle probabiliste", lambda: BayesianGaussianMixture(n_components=k, covariance_type="full", max_iter=200, random_state=SEED), True)
    add("Birch", "Hiérarchique incrémental", lambda: Birch(n_clusters=k, threshold=0.4), True)
    add("Hiérarchique (Ward)", "Hiérarchique", lambda: AgglomerativeClustering(n_clusters=k, linkage="ward"))
    add("Hiérarchique (average)", "Hiérarchique", lambda: AgglomerativeClustering(n_clusters=k, linkage="average"))
    add("Hiérarchique (complete)", "Hiérarchique", lambda: AgglomerativeClustering(n_clusters=k, linkage="complete"))
    add("Spectral", "Graphe", lambda: SpectralClustering(n_clusters=k, affinity="nearest_neighbors", n_neighbors=15,
        assign_labels="kmeans", random_state=SEED))
    add("Affinity Propagation", "Messages", lambda: AffinityPropagation(damping=0.9, max_iter=200, random_state=SEED), n=AP_N)
    add("Mean Shift", "Densité", lambda: MeanShift(bandwidth=BW, bin_seeding=True))
    add("DBSCAN", "Densité", lambda: DBSCAN(eps=EPS, min_samples=MIN_S))
    if HAS_HDBSCAN:
        add("HDBSCAN", "Densité", lambda: HDBSCAN(min_cluster_size=max(30, int(0.01 * len(Xe)))))
    add("OPTICS", "Densité", lambda: OPTICS(min_samples=max(10, int(0.004 * len(Xe))), xi=0.05, min_cluster_size=0.02))
    return A

ALGOS = algo_registry(K)
ref_labels = KMeans(n_clusters=K, n_init=N_INIT, random_state=SEED).fit_predict(Xe)

rows, labels_ev, idx_ev = [], {}, {}
for name, spec in ALGOS.items():
    t0 = time.time()
    try:
        if spec["n"] and spec["n"] < len(Xe):
            sub = np.sort(rng.choice(len(Xe), spec["n"], replace=False))
        else:
            sub = np.arange(len(Xe))
        lab = np.asarray(spec["make"]().fit_predict(Xe[sub]))
        labels_ev[name], idx_ev[name] = lab, sub
        ok = lab != -1
        ncl = len(set(lab[ok]))
        row = {"algorithme": name, "famille": spec["family"], "assignable": spec["assign"], "n_clusters": ncl,
               "bruit_%": (~ok).mean() * 100}
        if ncl >= 2 and ok.sum() > ncl:
            row.update({"silhouette": silhouette_score(Xe[sub][ok], lab[ok]),
                        "davies_bouldin": davies_bouldin_score(Xe[sub][ok], lab[ok]),
                        "calinski_harabasz": calinski_harabasz_score(Xe[sub][ok], lab[ok]),
                        "taille_min_%": np.bincount(lab[ok] - lab[ok].min()).min() / ok.sum() * 100,
                        "ARI vs K-Means": adjusted_rand_score(ref_labels[sub][ok], lab[ok])})
        row["temps (s)"] = time.time() - t0
        rows.append(row)
        print(f"  {name:28s} clusters = {ncl:3d}   silhouette = {row.get('silhouette', float('nan')):.3f}   ({row['temps (s)']:.1f} s)")
    except Exception as e:
        print(f"  {name:28s} ÉCHEC : {type(e).__name__}: {str(e)[:80]}")

algo_res = pd.DataFrame(rows).set_index("algorithme")
display(algo_res.sort_values("silhouette", ascending=False))
print(f"Paramètres des méthodes à densité : eps (DBSCAN) = {EPS:.3f}, min_samples = {MIN_S}, bande passante (Mean Shift) = {BW:.3f}")

## 8. Visualisation de la comparaison

In [ ]:
pca = PCA(n_components=2, random_state=SEED).fit(Xv)
names = list(labels_ev)
ncol = 4; nrow = int(np.ceil(len(names) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(5 * ncol, 4.2 * nrow))
for ax, name in zip(np.ravel(axes), names):
    Z = pca.transform(Xe[idx_ev[name]]); lab = labels_ev[name]
    noise = lab == -1
    ax.scatter(Z[noise, 0], Z[noise, 1], c="lightgrey", s=4)
    ax.scatter(Z[~noise, 0], Z[~noise, 1], c=lab[~noise], cmap="tab10", s=4, alpha=0.7)
    sil = algo_res.loc[name].get("silhouette", np.nan)
    ax.set_title(f"{name}\n{algo_res.loc[name, 'n_clusters']} clusters, silhouette {sil:.2f}", fontsize=9)
for ax in np.ravel(axes)[len(names):]:
    ax.axis("off")
plt.suptitle("Projection PCA des segments obtenus par chaque algorithme (échantillon)", y=1.0)
plt.tight_layout(); plt.show()

res_p = algo_res.dropna(subset=["silhouette"]).sort_values("silhouette")
fig, ax = plt.subplots(1, 2, figsize=(16, 0.4 * len(res_p) + 2))
ax[0].barh(res_p.index, res_p["silhouette"], color=["tab:blue" if a else "tab:grey" for a in res_p["assignable"]])
ax[0].set_title("Silhouette (bleu : peut affecter de nouveaux clients)")
res_p["ARI vs K-Means"].plot.barh(ax=ax[1], color="tab:green"); ax[1].set_title("Accord avec K-Means (ARI)")
plt.tight_layout(); plt.show()

## 9. Choix de l'algorithme final
Un segment n'est utile que si on peut **affecter de nouveaux clients** et s'il est assez gros pour justifier une action : on retient donc, parmi les algorithmes `assignable` dont **le plus petit segment pèse au moins `MIN_CLUSTER_SHARE` %**, celui de **silhouette maximale**. Ce garde-fou évite de retenir une solution qui isole quelques clients extrêmes et obtient une belle silhouette sans intérêt métier. `FINAL_ALGO` permet d'en forcer un autre. La **stabilité** de l'algorithme retenu est ensuite vérifiée sur l'ensemble des clients.

In [ ]:
assignable = algo_res[algo_res["assignable"] & algo_res["silhouette"].notna() & (algo_res["taille_min_%"] >= MIN_CLUSTER_SHARE)]
print(f"Algorithmes éligibles (assignables, plus petit segment >= {MIN_CLUSTER_SHARE} %) :", list(assignable.index))
final_name = FINAL_ALGO if FINAL_ALGO else (assignable["silhouette"].idxmax() if len(assignable) else "K-Means")
print("Algorithme retenu :", final_name)
display(algo_res.loc[[final_name]])

final_est = ALGOS[final_name]["make"]().fit(Xv)
raw = np.asarray(final_est.labels_ if hasattr(final_est, "labels_") else final_est.predict(Xv))
order = pd.Series(raw).value_counts().index.tolist()                 # numérotation par taille décroissante
m3["cluster"] = pd.Series(raw).map({old: new for new, old in enumerate(order)}).values
K_FINAL_USED = m3["cluster"].nunique()
print(m3["cluster"].value_counts().sort_index().to_string())

aris = []
for b in range(N_BOOT):
    sub = rng.choice(len(Xv), size=int(0.8 * len(Xv)), replace=False)
    est_b = clone(ALGOS[final_name]["make"]()).fit(Xv[sub])
    pred_b = est_b.labels_ if (len(sub) == len(Xv)) else est_b.predict(Xv)
    aris.append(adjusted_rand_score(raw, pred_b))
print(f"Stabilité de {final_name} (ARI moyen sur {N_BOOT} sous-échantillons de 80 %) : {np.mean(aris):.3f}")

## 10. Visualisation des segments retenus

In [ ]:
v_idx = rng.choice(len(Xv), size=min(8000, len(Xv)), replace=False)
Z = pca.transform(Xv[v_idx])
fig, ax = plt.subplots(1, 3, figsize=(21, 5.5))
sc = ax[0].scatter(Z[:, 0], Z[:, 1], c=m3["cluster"].values[v_idx], cmap="tab10", s=6, alpha=0.6)
ax[0].set_title(f"Projection PCA (variance expliquée {pca.explained_variance_ratio_.sum()*100:.0f} %)")
ax[0].legend(*sc.legend_elements(), title="cluster")
z = (m3.groupby("cluster")[CLUSTER_FEATURES].mean() - m3[CLUSTER_FEATURES].mean()) / m3[CLUSTER_FEATURES].std()
sns.heatmap(z, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax[1]); ax[1].set_title("Écart à la moyenne globale (en écarts-types)")
m3["cluster"].value_counts().sort_index().plot.bar(ax=ax[2], color="tab:blue"); ax[2].set_title("Taille des segments"); ax[2].tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.show()

## 11. Profils des segments

In [ ]:
def top(s): return s.value_counts().index[0]
prof = m3.groupby("cluster").agg(
    clients=("customer_unique_id", "count"), recence_mediane_j=("recency_days", "median"),
    montant_median=("monetary_total", "median"), panier_moyen=("avg_order_value", "mean"),
    note_moyenne=("avg_review_score", "mean"), duree_livraison_moy_j=("avg_delivery_days", "mean"),
    part_en_retard=("late_share", "mean"), retard_max_moy_j=("max_days_late", "mean"),
    ecart_promesse_moy_j=("avg_gap_days", "mean"), part_reachat=("is_repeat_customer", "mean"),
    part_sans_avis=("review_missing", "mean"), region_dominante=("region", top), categorie_dominante=("last_category", top))
prof.insert(1, "part_clients_%", prof["clients"] / len(m3) * 100)
display(prof.round(2))
print("Répartition des régions par segment (% de la région) :")
display((pd.crosstab(m3["region"], m3["cluster"], normalize="index") * 100).round(1))

## 12. Noms métier et actions recommandées
Noms **proposés automatiquement** à partir des écarts à la moyenne (récence, valeur, satisfaction, livraison), à **relire et corriger** : `LABEL_OVERRIDE` et `ACTION_OVERRIDE` permettent de les modifier.

In [ ]:
LABEL_OVERRIDE, ACTION_OVERRIDE = {}, {}       # par exemple {0: "Clients satisfaits à faible valeur"}
cols_z = ["recency_days", "log_monetary", "avg_review_score", "avg_delivery_days", "late_share"]
zf = (m3.groupby("cluster")[cols_z].mean() - m3[cols_z].mean()) / m3[cols_z].std()

def name_and_action(c):
    r = zf.loc[c]
    parts = ["valeur élevée" if r["log_monetary"] > 0.5 else "petits paniers" if r["log_monetary"] < -0.5 else "valeur moyenne"]
    if r["recency_days"] < -0.5: parts.append("récents")
    elif r["recency_days"] > 0.5: parts.append("anciens")
    unhappy, happy = r["avg_review_score"] < -0.5, r["avg_review_score"] > 0.3
    delivery_issue = (r["late_share"] > 0.5) or (r["avg_delivery_days"] > 0.5)
    if unhappy: parts.append("insatisfaits")
    elif happy: parts.append("satisfaits")
    if delivery_issue: parts.append("livraison longue ou en retard")
    label = ", ".join(parts).capitalize()
    if unhappy and delivery_issue:
        action = "Geste commercial et message d'excuse ; suivi prioritaire des vendeurs et trajets concernés"
    elif unhappy:
        action = "Enquête de satisfaction ; vérifier la qualité du produit ou du vendeur"
    elif r["recency_days"] > 0.5:
        action = "Campagne de réactivation à faible coût"
    elif happy and r["recency_days"] < -0.5:
        action = "Programme de fidélité et offres de réachat"
    elif delivery_issue:
        action = "Communication proactive sur les délais ; surveillance logistique"
    else:
        action = "Communication standard ; suivre l'évolution du segment"
    return label, action

rows = []
for c in sorted(m3["cluster"].unique()):
    label, action = name_and_action(c)
    rows.append({"cluster": c, "nom": LABEL_OVERRIDE.get(c, label), "action recommandée": ACTION_OVERRIDE.get(c, action),
                 "clients": int((m3["cluster"] == c).sum()), "part_%": (m3["cluster"] == c).mean() * 100})
segments = pd.DataFrame(rows).set_index("cluster")
pd.set_option("display.max_colwidth", 120)
display(segments)
m3["segment"] = m3["cluster"].map(segments["nom"])

## 13. Vue métier : retard vécu et satisfaction par segment

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 5))
g = prof.reset_index()
ax[0].scatter(g["part_en_retard"] * 100, g["note_moyenne"], s=g["clients"] / g["clients"].max() * 1500 + 50, alpha=0.6)
for _, r in g.iterrows():
    ax[0].annotate(f"{int(r['cluster'])}", (r["part_en_retard"] * 100, r["note_moyenne"]), ha="center", va="center", fontweight="bold")
ax[0].set_xlabel("Part de commandes en retard (%)"); ax[0].set_ylabel("Note moyenne")
ax[0].set_title("Retard vécu et satisfaction (taille = nombre de clients)")
late_vs_ok = m3.assign(vecu_retard=np.where(m3["late_share"] > 0, "au moins un retard", "jamais en retard")) \
               .groupby("vecu_retard").agg(clients=("customer_unique_id", "count"), note=("avg_review_score", "mean"),
                                           montant_median=("monetary_total", "median"))
display(late_vs_ok)
late_vs_ok["note"].plot.bar(ax=ax[1], color=["tab:red", "tab:green"]); ax[1].set_title("Note moyenne selon le retard vécu")
ax[1].tick_params(axis="x", rotation=0); ax[1].set_ylim(0, 5)
plt.tight_layout(); plt.show()

high_value = m3["monetary_total"] >= m3["monetary_total"].quantile(0.75)
at_risk = m3[high_value & (m3["late_share"] > 0) & (m3["avg_review_score"] <= 2)]
print(f"Clients à forte valeur (25 % supérieurs), touchés par un retard et notés 2 ou moins : {len(at_risk):,} ({len(at_risk)/len(m3)*100:.2f} % des clients)")

## 14. Sauvegarde

In [ ]:
import joblib
cols = ["customer_unique_id", "cluster", "segment", "customer_state", "region"] + CLUSTER_FEATURES
m3[cols].to_csv(os.path.join(OUT_DIR, "customer_segments.csv"), index=False)
prof.to_csv(os.path.join(OUT_DIR, "segment_profiles.csv"))
segments.to_csv(os.path.join(OUT_DIR, "segment_names_actions.csv"))
algo_res.to_csv(os.path.join(OUT_DIR, "model3_all_algorithms.csv"))
joblib.dump({"estimator": final_est, "features": RS, "clip_low": lo.to_dict(), "clip_high": hi.to_dict(),
             "cluster_order": {int(old): int(new) for new, old in enumerate(order)}},
            os.path.join(OUT_DIR, "model3_final_clusterer.joblib"))

summary = {"algorithm": final_name, "k": int(K_FINAL_USED), "features_scaled": RS, "clip_quantile": CLIP_Q,
           "k_selection": sel.reset_index().to_dict("records"),
           "algorithm_comparison": algo_res.reset_index().replace({np.nan: None}).to_dict("records"),
           "sizes": {int(k): int(v) for k, v in m3["cluster"].value_counts().sort_index().items()}}
with open(os.path.join(OUT_DIR, "model3_summary.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2, default=float)
print("Fichiers :", sorted(os.listdir(OUT_DIR)))

## Synthèse du modèle 3
À compléter avec tes résultats :
- **Choix de *k*** : silhouette, Davies-Bouldin, stabilité et BIC (cellules 4 à 6).
- **Comparaison des algorithmes** : lesquels trouvent des groupes proches de K-Means (ARI élevé), lesquels isolent du bruit ou donnent un nombre de groupes très différent (cellules 7 et 8).
- **Algorithme retenu** et sa stabilité (cellule 9).
- **Profils, noms et actions** de chaque segment (cellules 11 et 12).
- **Limite à écrire dans le rapport** : 97 % de clients à achat unique ; la segmentation décrit des profils d'expérience, pas une fidélité mesurée. Des scores de silhouette modérés sont attendus (variables continues sans frontières nettes).

Étape suivante : phases 5 (Evaluation) et 6 (Deployment) : tableau de bord alimenté par `model1_predictions.csv`, `model2_predictions.csv` et `customer_segments.csv`.